<a href="https://colab.research.google.com/github/Xinerrevis/Ibuprofen/blob/main/exploring_2_git.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import os
import requests
import time




In [ ]:
os.environ['NCBI_API_KEY'] = #Insert your NCBI API KEY here! ""
from metapub import FindIt

In [ ]:

def download_with_metapub_findit(csv_path, output_dir):
    """
    Downloads PDFs utilizing dynamic publisher-specific URL resolution via metapub.
    """
    os.makedirs(output_dir, exist_ok=True)
    df = pd.read_csv(csv_path)
    missing_data = []

    for index, row in df.iterrows():
        # Ensure PMID is formatted as a clean integer string
        pmid = str(row['pmid']).split('.')[0] if pd.notna(row['pmid']) else None

        if not pmid or pmid == 'nan':
            missing_data.append({'title': row.get('title'), 'reason': 'No PMID provided in CSV'})
            continue

        try:
            # Initialize FindIt with PMID to execute the "publisher dance"
            src = FindIt(pmid)

            if src.url:
                # A direct, unpaywalled PDF link was dynamically constructed
                file_path = os.path.join(output_dir, f"PMID_{pmid}.pdf")

                # Execute GET request and stream the binary download
                response = requests.get(src.url, stream=True, timeout=30)

                # Verify HTTP status and MIME type
                content_type = response.headers.get('Content-Type', '')
                if response.status_code == 200 and 'application/pdf' in content_type:
                    with open(file_path, 'wb') as f:
                        for chunk in response.iter_content(chunk_size=2048):
                            f.write(chunk)
                    print(f" PMID {pmid} - PDF saved.")
                else:
                    reason = f"HTTP {response.status_code} or Invalid Content-Type: {content_type}"
                    missing_data.append({'pmid': pmid, 'reason': reason})
                    print(f" {pmid} - {reason}")
            else:
                # FindIt could not resolve a legal OA link (e.g., behind a hard paywall)
                print(f" PMID {pmid} - Reason: {src.reason}")
                missing_data.append({'pmid': pmid, 'reason': src.reason})

        except Exception as e:
            print(f" Metapub failure for PMID {pmid}: {e}")
            missing_data.append({'pmid': pmid, 'reason': str(e)})

        # Metapub handles NCBI rate limiting natively if API key is provided,
        # but a small delay ensures courtesy to third-party publisher servers.
        time.sleep(1.5)

    # Save missing log for audit purposes
    if missing_data:
        error_path = os.path.join(output_dir, 'metapub_missing_audit.csv')
        pd.DataFrame(missing_data).to_csv(error_path, index=False)
        print(f"\n Metapub failure log saved to {error_path}")

# Example Execution
# download_with_metapub_findit('example_output.csv', './PDF_Corpus_Metapub')

In [ ]:
# download_with_metapub_findit('example_output.csv', './PDF_Corpus_Metapub')